In [1]:
import pandas as pd
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.svm import SVC

In [2]:
df=pd.read_csv('loan_data.csv')

In [3]:
df

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44995,27.0,male,Associate,47971.0,6,RENT,15000.0,MEDICAL,15.66,0.31,3.0,645,No,1
44996,37.0,female,Associate,65800.0,17,RENT,9000.0,HOMEIMPROVEMENT,14.07,0.14,11.0,621,No,1
44997,33.0,male,Associate,56942.0,7,RENT,2771.0,DEBTCONSOLIDATION,10.02,0.05,10.0,668,No,1
44998,29.0,male,Bachelor,33164.0,4,RENT,12000.0,EDUCATION,13.23,0.36,6.0,604,No,1


In [4]:
x = df.drop(columns='loan_status')
y = df.loan_status

In [5]:
xtrain, xtest, ytrain, ytest = train_test_split(x, y, random_state=42, train_size=0.8)

In [6]:
# num_col = x.select_dtypes(include = 'number').columns
obj_col = x.select_dtypes(include = 'object').columns

In [7]:
x[obj_col].nunique()

person_gender                     2
person_education                  5
person_home_ownership             4
loan_intent                       6
previous_loan_defaults_on_file    2
dtype: int64

In [8]:
x['person_education'].unique()

array(['Master', 'High School', 'Bachelor', 'Associate', 'Doctorate'],
      dtype=object)

In [9]:
order = ['Master', 'High School', 'Bachelor', 'Associate', 'Doctorate']

In [10]:
preprocessing  = ColumnTransformer(
    transformers = [
        ('onehot_encodedr', OneHotEncoder(handle_unknown='ignore'), obj_col.drop('person_education')),
        ('orndinal_encoder', OrdinalEncoder(categories=[order], handle_unknown='use_encoded_value', unknown_value=-1), ['person_education'])

    ], remainder= 'passthrough'
)

main_pipeline = Pipeline(
    steps=[
        ('preprocessing', preprocessing),
        ('model', SVC(random_state=42))
    ]
)

grid_search_cv = GridSearchCV(
    estimator = main_pipeline,
    kernal={
        'model__criterion' : ['gini', 'entropy'],           
        'model__min_samples_split' : [2, 5, 7, 10],
        'model__min_samples_leaf' : [1, 3, 5, 7, 10],
        'model__splitter' : ['best', 'random']
    }, verbose = 10, n_jobs = -1
)

grid_search_cv.fit(xtrain, ytrain)

TypeError: GridSearchCV.__init__() got an unexpected keyword argument 'kernal'

 if we are using any algorithm then there is no need to use that model name but if we are using pipeline then we have to use model name along with double underscore
e.g., model__parameter

In [ ]:
grid_search_cv.best_estimator_

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('onehot_encodedr', ...), ('orndinal_encoder', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
grid_search_cv.best_params_

{'model__criterion': 'entropy',
 'model__max_depth': 20,
 'model__min_samples_leaf': 10,
 'model__min_samples_split': 2,
 'model__splitter': 'best'}

In [ ]:
grid_search_cv.cv_results_

{'mean_fit_time': array([0.40701299, 0.1784955 , 0.38342366, 0.17366505, 0.36153522,
        0.16847849, 0.35575862, 0.15776482, 0.34637327, 0.16058426,
        0.34450169, 0.1685462 , 0.35215263, 0.16504211, 0.34127235,
        0.16264215, 0.34793739, 0.15915008, 0.34733725, 0.15710764,
        0.35235481, 0.15944934, 0.34076433, 0.15539474, 0.32818174,
        0.153824  , 0.33816895, 0.15361743, 0.32778511, 0.16195202,
        0.33544297, 0.16394219, 0.33708119, 0.15642996, 0.37343612,
        0.1732224 , 0.41135283, 0.17093205, 0.3279644 , 0.15651622,
        0.1934278 , 0.11888232, 0.19994717, 0.12713852, 0.19475169,
        0.12503486, 0.21519227, 0.12760892, 0.18888731, 0.12295985,
        0.18335681, 0.12832232, 0.19540267, 0.13587413, 0.19485426,
        0.10896344, 0.18542061, 0.12581997, 0.19289536, 0.11047449,
        0.19479179, 0.11958923, 0.17732654, 0.12072206, 0.18831449,
        0.11794615, 0.18455038, 0.13097801, 0.20128646, 0.12163997,
        0.18908792, 0.10372815,

In [ ]:
res = pd.DataFrame(grid_search_cv.cv_results_)

In [ ]:
res.sort_values(by='rank_test_score')

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__criterion,param_model__max_depth,param_model__min_samples_leaf,param_model__min_samples_split,param_model__splitter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
316,0.448780,0.018876,0.027518,0.001975,entropy,20,10,7,best,"{'model__criterion': 'entropy', 'model__max_de...",0.909028,0.916111,0.914028,0.915694,0.915972,0.914167,0.002676,1
318,0.359790,0.028460,0.030524,0.002537,entropy,20,10,10,best,"{'model__criterion': 'entropy', 'model__max_de...",0.909028,0.916111,0.914028,0.915694,0.915972,0.914167,0.002676,1
312,0.382793,0.021422,0.032313,0.002156,entropy,20,10,2,best,"{'model__criterion': 'entropy', 'model__max_de...",0.909028,0.916111,0.914028,0.915694,0.915972,0.914167,0.002676,1
314,0.377659,0.028532,0.042614,0.015778,entropy,20,10,5,best,"{'model__criterion': 'entropy', 'model__max_de...",0.909028,0.916111,0.914028,0.915694,0.915972,0.914167,0.002676,1
78,0.188288,0.015739,0.028380,0.002670,gini,5,10,10,best,"{'model__criterion': 'gini', 'model__max_depth...",0.909861,0.916667,0.908889,0.915556,0.915417,0.913278,0.003231,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71,0.103728,0.004863,0.023601,0.004791,gini,5,7,10,random,"{'model__criterion': 'gini', 'model__max_depth...",0.873056,0.872222,0.871667,0.872500,0.872639,0.872417,0.000461,393
79,0.115671,0.010444,0.029227,0.001680,gini,5,10,10,random,"{'model__criterion': 'gini', 'model__max_depth...",0.873056,0.872222,0.871667,0.872500,0.872222,0.872333,0.000451,397
77,0.106520,0.012769,0.024564,0.002834,gini,5,10,7,random,"{'model__criterion': 'gini', 'model__max_depth...",0.873056,0.872222,0.871667,0.872500,0.872222,0.872333,0.000451,397
75,0.125084,0.015860,0.031362,0.004411,gini,5,10,5,random,"{'model__criterion': 'gini', 'model__max_depth...",0.873056,0.872222,0.871667,0.872500,0.872222,0.872333,0.000451,397


## marginal distance -->